####  1) Código – Imports e configuração
Nesta célula:
- Carregamos bibliotecas (`pandas`, `geopandas`, `shapely`, `folium`) e definimos sistemas de referência (`EPSG:20790` métrico; `EPSG:4326` WGS84 para mapas).
Teoria: manter CRS consistente evita erros em distâncias e mapas; Leaflet/Folium opera em WGS84.


In [7]:
# 1) Imports e configuração
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import folium
from folium.plugins import MiniMap

# CRS: usar Lisboa_Hayford_Gauss_IGeoE (EPSG:20790) para coords em metros
METRIC_CRS = 20790
WGS84 = 4326

print("[Init] Bibliotecas carregadas. CRS métrico=EPSG:20790 (Lisboa_Hayford_Gauss_IGeoE), mapa em WGS84=EPSG:4326")


[Init] Bibliotecas carregadas. CRS métrico=EPSG:20790 (Lisboa_Hayford_Gauss_IGeoE), mapa em WGS84=EPSG:4326


####  2) Código – Leitura de dados e inspeção inicial
Nesta célula:
- Lemos os CSVs com parsing de datas (`piezo`, `condut`, `caudal`, `meteo`).
- Imprimimos dimensões e colunas para verificar consistência e nomes.
Teoria: garantir que as datas são interpretadas corretamente e que as colunas necessárias existem.


In [8]:
# 2) Leitura dos CSVs (DataFrames)
# Paths absolutos conforme pedido
base = "/Users/diogopinto/Documents/Pessoal/path_4med/eobs_netcdf/usar_model/correlacao_matriz"

piezo_csv  = f"{base}/piezo_tejo_loc_zvt.csv"
condut_csv = f"{base}/condut_tejo_loc_zvt.csv"
caudal_csv = f"{base}/caudal_tejo_loc.csv"
meteo_csv  = f"{base}/meteo.csv"

# Lê com parse de datas
piezo_df  = pd.read_csv(piezo_csv,  parse_dates=["data"], dayfirst=False)
condut_df = pd.read_csv(condut_csv, parse_dates=["data"], dayfirst=False)
caudal_df = pd.read_csv(caudal_csv, parse_dates=["data"], dayfirst=False)
meteo_df  = pd.read_csv(meteo_csv,  parse_dates=["time"], dayfirst=False)

print("[Leitura] piezo_df:", piezo_df.shape, "condut_df:", condut_df.shape, "caudal_df:", caudal_df.shape, "meteo_df:", meteo_df.shape)

# Conferir colunas relevantes
print("piezo_df cols:", list(piezo_df.columns)[:12])
print("condut_df cols:", list(condut_df.columns)[:12])
print("caudal_df cols:", list(caudal_df.columns)[:12])
print("meteo_df cols:", list(meteo_df.columns)[:12])


[Leitura] piezo_df: (17308, 12) condut_df: (996, 12) caudal_df: (17805, 7) meteo_df: (1297460, 7)
piezo_df cols: ['id', 'data', 'codigo', 'nivel_piezometrico', 'profundidade_nivel_agua', 'coord_x_m', 'coord_y_m', 'altitude_m', 'sistema_aquifero', 'estado', 'freguesia', 'created_at']
condut_df cols: ['id', 'data', 'codigo', 'condutividade', 'condcamp20c', 'coord_x_m', 'coord_y_m', 'altitude_m', 'sistema_aquifero', 'estado', 'freguesia', 'created_at']
caudal_df cols: ['id', 'data', 'localizacao', 'caudal_médio_diário(m3/s)', 'coord_x_m', 'coord_y_m', 'created_at']
meteo_df cols: ['time', 'tx', 'tn', 'rr', 'lat', 'long', 'ETo']


####  3) Código – Criação de GeoDataFrames e reprojeções
Nesta célula:
- Transformamos os DataFrames em GeoDataFrames no CRS métrico (`EPSG:20790`) usando `coord_x_m`/`coord_y_m`.
- Para meteo (E‑OBS) criamos geometria a partir de `lat/long` (WGS84) e depois reprojetamos para o CRS métrico.
- Deduplicamos pontos por identificador (poço/estação) para evitar pontos repetidos.


In [9]:
# 3) Criar GeoDataFrames (coordenadas em metros -> METRIC_CRS)
# Pelas colunas "coord_x_m" e "coord_y_m", assumimos EPSG:3763

def df_to_gdf(df, x_col="coord_x_m", y_col="coord_y_m", crs=METRIC_CRS):
    has_coords = x_col in df.columns and y_col in df.columns
    if not has_coords:
        raise ValueError(f"Colunas {x_col},{y_col} não encontradas no DF")
    geometry = gpd.points_from_xy(df[x_col], df[y_col], crs=crs)
    return gpd.GeoDataFrame(df.copy(), geometry=geometry, crs=crs)

piezo_gdf  = df_to_gdf(piezo_df)
condut_gdf = df_to_gdf(condut_df)
caudal_gdf = df_to_gdf(caudal_df)

# Meteo tem colunas lat/long (já em WGS84). Criamos GDF em 4326, depois levamos para métrico se quisermos.
if set(["lat", "long"]).issubset(meteo_df.columns):
    meteo_gdf = gpd.GeoDataFrame(
        meteo_df.copy(),
        geometry=gpd.points_from_xy(meteo_df["long"], meteo_df["lat"], crs=WGS84)
    ).to_crs(METRIC_CRS)
else:
    meteo_gdf = None

print("[GDF] piezo:", piezo_gdf.crs, "condut:", condut_gdf.crs, "caudal:", caudal_gdf.crs, "meteo:", None if meteo_gdf is None else meteo_gdf.crs)

# Deduplcar pontos por identificador
piezo_unique  = piezo_gdf.drop_duplicates(subset=["codigo"])
condut_unique = condut_gdf.drop_duplicates(subset=["codigo"]) if "codigo" in condut_gdf.columns else condut_gdf
caudal_unique = caudal_gdf.drop_duplicates(subset=["localizacao"]) if "localizacao" in caudal_gdf.columns else caudal_gdf
meteo_unique  = None if meteo_gdf is None else meteo_gdf.drop_duplicates(subset=["geometry"])


[GDF] piezo: EPSG:20790 condut: EPSG:20790 caudal: EPSG:20790 meteo: EPSG:20790


####  4) Código – Mapa interativo dos pontos (Folium)
Nesta célula:
- Reprojetamos para WGS84 (EPSG:4326) e construímos um mapa com camadas: poços (piezo), condutividade, caudal e meteo.
- O mapa permite inspeção visual e navegação por camadas.
Teoria: inspeção espacial auxilia a contextualização das correlações e distâncias.


In [10]:
# 4) Mapa Leaflet (Folium) em WGS84
# Converter para WGS84
piezo_wgs   = piezo_unique.to_crs(WGS84)
condut_wgs  = condut_unique.to_crs(WGS84)
caudal_wgs  = caudal_unique.to_crs(WGS84)
meteo_wgs   = None if meteo_unique is None else meteo_unique.to_crs(WGS84)

# Centro do mapa: média dos poços se existirem
if len(piezo_wgs) > 0:
    center_lat = float(piezo_wgs.geometry.y.mean())
    center_lon = float(piezo_wgs.geometry.x.mean())
else:
    center_lat, center_lon = 39.5, -8.0

m = folium.Map(location=[center_lat, center_lon], zoom_start=8, tiles='OpenStreetMap')
MiniMap(toggle_display=True).add_to(m)

# FeatureGroups
fg_piezo = folium.FeatureGroup(name='Poços (piezo)').add_to(m)
fg_cond  = folium.FeatureGroup(name='Condutividade').add_to(m)
fg_caud  = folium.FeatureGroup(name='Caudal').add_to(m)
fg_met   = folium.FeatureGroup(name='Meteo (E-OBS)').add_to(m)

# Poços
if "codigo" in piezo_wgs.columns:
    for _, r in piezo_wgs[["codigo","geometry"]].drop_duplicates("codigo").iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=3, color='blue', fill=True, fill_opacity=0.7,
            popup=f"Poço {r['codigo']}"
        ).add_to(fg_piezo)
else:
    for _, r in piezo_wgs.iterrows():
        folium.CircleMarker([r.geometry.y, r.geometry.x], radius=3, color='blue', fill=True, fill_opacity=0.7).add_to(fg_piezo)

# Condutividade (usa 'codigo' também)
if "codigo" in condut_wgs.columns:
    for _, r in condut_wgs[["codigo","geometry"]].drop_duplicates("codigo").iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=3, color='green', fill=True, fill_opacity=0.7,
            popup=f"Condut {r['codigo']}"
        ).add_to(fg_cond)
else:
    for _, r in condut_wgs.iterrows():
        folium.CircleMarker([r.geometry.y, r.geometry.x], radius=3, color='green', fill=True, fill_opacity=0.7).add_to(fg_cond)

# Caudal (usa 'localizacao')
if "localizacao" in caudal_wgs.columns:
    for _, r in caudal_wgs[["localizacao","geometry"]].drop_duplicates("localizacao").iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=3, color='red', fill=True, fill_opacity=0.7,
            popup=f"Caudal {r['localizacao']}"
        ).add_to(fg_caud)
else:
    for _, r in caudal_wgs.iterrows():
        folium.CircleMarker([r.geometry.y, r.geometry.x], radius=3, color='red', fill=True, fill_opacity=0.7).add_to(fg_caud)

# Meteo: muitos pontos idênticos, desenhar únicos
if meteo_wgs is not None:
    _meteo_unique_pts = meteo_wgs[["geometry"]].drop_duplicates().reset_index(drop=True)
    for _, r in _meteo_unique_pts.iterrows():
        folium.CircleMarker(
            location=[r.geometry.y, r.geometry.x],
            radius=1, color='orange', fill=True, fill_opacity=0.5
        ).add_to(fg_met)

folium.LayerControl(collapsed=False).add_to(m)

m


####  5) Código – Distâncias aos pontos de caudal/condutividade/meteo
Nesta célula:
- Para cada poço, encontramos a estação de caudal, a estação de condutividade e o grid E‑OBS mais próximos (em CRS métrico) via `sjoin_nearest`.
- Calculamos distâncias (km) e geramos uma tabela larga por poço.
- Exportamos `distancias_outros_para_pocos.csv` para reutilização nas agregações mensais.
Teoria: a proximidade espacial aumenta a relevância da série externa para o poço; registamos distâncias para análise posterior.


In [11]:
# 5) Distâncias por poço (largura): caudal/condut/eobs mais próximos e exportar CSV
import numpy as np

# Garante CRS métrico
piezo_m   = piezo_unique.to_crs(METRIC_CRS)
condut_m  = condut_unique.to_crs(METRIC_CRS)
caudal_m  = caudal_unique.to_crs(METRIC_CRS)
meteo_m   = None if meteo_unique is None else meteo_unique.to_crs(METRIC_CRS)

# Função auxiliar: para cada poço, obter estação mais próxima de um conjunto

def nearest_wide(target_gdf, id_col, station_col_name, dist_col_name):
    if target_gdf is None or len(target_gdf) == 0:
        left = piezo_m[["codigo","geometry"]].rename(columns={"codigo":"poco_id"}).copy()
        out = left.drop(columns=["geometry"]).copy()
        out[station_col_name] = np.nan
        out[dist_col_name] = np.nan
        return out

    left = piezo_m[["codigo","geometry"]].rename(columns={"codigo":"poco_id"})

    joined = gpd.sjoin_nearest(
        left,
        target_gdf[["geometry"]],
        how="left",
        distance_col="distance_m",
    )

    # Mapear identificador da estação do índice da direita
    if id_col is not None and id_col in target_gdf.columns:
        idx_to_id = target_gdf.reset_index().set_index("index")[id_col]
        station_vals = joined["index_right"].map(idx_to_id)
    else:
        if target_gdf is not None and set(["lat","long"]).issubset(target_gdf.columns):
            right_idx = target_gdf.reset_index().set_index("index")
            grid_id_series = right_idx["lat"].round(4).astype(str) + "," + right_idx["long"].round(4).astype(str)
            station_vals = joined["index_right"].map(grid_id_series)
        else:
            station_vals = joined["index_right"]

    out = pd.DataFrame({
        "poco_id": joined["poco_id"].values,
        station_col_name: station_vals.values,
        dist_col_name: (joined["distance_m"].astype(float) / 1000.0).values,
    })
    return out

# Construir tabela larga por poço
wide = nearest_wide(caudal_m, id_col="localizacao", station_col_name="caudal_station", dist_col_name="dist_km_caudal")
wide = wide.merge(
    nearest_wide(condut_m, id_col="codigo", station_col_name="cond_station", dist_col_name="dist_km_condut"),
    on="poco_id", how="left"
)
if meteo_m is not None:
    wide = wide.merge(
        nearest_wide(meteo_m, id_col=None, station_col_name="eobs_grid", dist_col_name="dist_km_eobs"),
        on="poco_id", how="left"
    )

# Ordenar por id do poço para legibilidade
wide = wide.sort_values("poco_id").reset_index(drop=True)

out_csv = f"{base}/distancias_outros_para_pocos.csv"
wide.to_csv(out_csv, index=False)

print(f"[Export] Linhas: {len(wide)} -> {out_csv}")
wide.head(10)


[Export] Linhas: 73 -> /Users/diogopinto/Documents/Pessoal/path_4med/eobs_netcdf/usar_model/correlacao_matriz/distancias_outros_para_pocos.csv


,poco_id,caudal_station,dist_km_caudal,cond_station,dist_km_condut,eobs_grid,dist_km_eobs
0,329/6,ALMOUROL (17G/02H),13.797998,329/6,0.000000,"39.45,-8.55",5.461122
1,330/183,ALMOUROL (17G/02H),10.183768,330/235,0.548179,"39.45,-8.45",5.205765
2,331/2,ALMOUROL (17G/02H),9.665531,331/1,0.083934,"39.45,-8.25",3.542648
3,341/17,ALMOUROL (17G/02H),18.105010,341/17,0.000000,"39.35,-8.55",1.506999
4,342/78,ALMOUROL (17G/02H),15.622714,341/254,2.932865,"39.35,-8.45",2.837661
5,342/97,ALMOUROL (17G/02H),13.717825,342/115,2.071835,"39.35,-8.45",3.063060
6,353/346,ALMOUROL (17G/02H),23.682476,353/373,0.001000,"39.25,-8.55",4.464923
7,353/353,ALMOUROL (17G/02H),28.458410,353/687,0.862165,"39.25,-8.55",1.073309
8,353/687,ALMOUROL (17G/02H),28.771478,353/687,0.000000,"39.25,-8.55",0.951226
9,365/1076,ALMOUROL (17G/02H),37.549224,365/27,1.069766,"39.15,-8.55",2.428582


####  6) Código – Agregações mensais e heatmaps de correlação por poço
Nesta célula construímos o dataset mensal unificado por poço e geramos heatmaps de correlação.

- Preparação e agregações mensais
  - Criamos a chave mensal `ym` (ano‑mês) em todas as tabelas.
  - Piezo (por `codigo`+`ym`):
    - `profundidade_mediana` = mediana mensal de `profundidade_nivel_agua`
    - `nivel_piezometrico_mediana` = mediana mensal de `nivel_piezometrico`
  - Condutividade (por `codigo`+`ym`):
    - `condutividade_median` = mediana mensal, usando `condutividade` com fallback para `condcamp20c`
  - Caudal (por `localizacao`+`ym`):
    - `caudal_medio_mensal` = média mensal da coluna de caudal
  - Meteo (por `lat,long`+`ym`):
    - `tx_mean`, `tn_mean`, `eto_mean` = médias mensais; `rr_sum` = soma mensal
    - Criamos `eobs_grid` (lat,long arredondados) para compatibilizar com o grid mais próximo do poço

- Mapeamento para o poço certo
  - Usamos a tabela de proximidades da célula 5 (`distancias_outros_para_pocos.csv`) para, por `codigo`:
    - juntar o caudal de `caudal_station`, a condutividade de `cond_station` e a meteo de `eobs_grid`.
  - Resultado: `unified` com uma linha por poço e mês.

- Variáveis incluídas nos heatmaps (por poço)
  - `rr_sum`, `profundidade_mediana`, `eto_mean`, `caudal_medio_mensal`, `condutividade_median`
  - (Calculadas mas não entram no heatmap: `tx_mean`, `tn_mean`)

- Cálculo das correlações e imagens
  - Para cada poço, selecionamos as colunas acima e indexamos por `ym`.
  - Removemos colunas totalmente vazias.
  - Calculamos a matriz de correlação com `min_periods=10`:
    - Pearson (linear) e Spearman (monotónica).
  - Geramos e guardamos imagens dos heatmaps em `heatmaps/pearson/` e `heatmaps/spearman/`.

- Saídas
  - CSV mensal unificado: `mensal_por_poco.csv`.
  - Heatmaps por poço: ficheiros `heatmap_<metodo>_<poco>.png` nas pastas referidas.

Notas
- Os heatmaps usam valores do mesmo mês (sem lags). Lags foram analisados separadamente nas secções 8.1/8.2.
- Se um poço tiver <10 meses válidos para um par de variáveis, a correlação fica NaN para esse par.


In [12]:
# 6) Agregações mensais e dataset unificado por poço + heatmap de correlação
import numpy as np
import pandas as pd

# Carregar mapeamento de estações mais próximas (criado na célula 5) com fallback de caminho
from pathlib import Path
nearest_csv = f"{base}/distancias_outros_para_pocos.csv"
alt_nearest_csv = f"{base}/output/distancias_outros_para_pocos.csv"
csv_path = nearest_csv if Path(nearest_csv).exists() else alt_nearest_csv
nearest_df = pd.read_csv(csv_path).rename(columns={
    "poco_id":"codigo",
    "caudal_station":"caudal_station",
    "cond_station":"cond_station",
    "eobs_grid":"eobs_grid"
}).drop_duplicates("codigo")

# Preparar chaves por poço
piezo_keys = piezo_df[["codigo"]].drop_duplicates().merge(
    nearest_df[["codigo","caudal_station","cond_station","eobs_grid"]],
    on="codigo", how="left"
)

# Criar chave mensal
piezo_df["ym"]  = piezo_df["data"].dt.to_period("M").dt.to_timestamp()
condut_df["ym"] = condut_df["data"].dt.to_period("M").dt.to_timestamp()
caudal_df["ym"] = caudal_df["data"].dt.to_period("M").dt.to_timestamp()
meteo_df["ym"]  = meteo_df["time"].dt.to_period("M").dt.to_timestamp()

# Agregações mensais segundo regras propostas
# - Piezo: medianas
piezo_month = (piezo_df
    .groupby(["codigo","ym"], as_index=False)
    .agg(
        profundidade_mediana=("profundidade_nivel_agua","median"),
        nivel_piezometrico_mediana=("nivel_piezometrico","median")
    )
)

# - Condutividade: mediana mensal (usar 'condutividade' e fallback para 'condcamp20c')
if "condutividade" in condut_df.columns:
    condut_df["condutividade"] = pd.to_numeric(condut_df["condutividade"], errors="coerce")
if "condcamp20c" in condut_df.columns:
    condut_df["condcamp20c"] = pd.to_numeric(condut_df["condcamp20c"], errors="coerce")
condut_df["cond_merge"] = condut_df.get("condutividade", pd.Series(index=condut_df.index)).where(
    condut_df.get("condutividade", pd.Series(index=condut_df.index)).notna(),
    condut_df.get("condcamp20c", pd.Series(index=condut_df.index))
)
condut_month = (condut_df
    .groupby(["codigo","ym"], as_index=False)
    .agg(condutividade_median=("cond_merge","median"))
)

# - Caudal: média mensal
_caudal_col = [c for c in caudal_df.columns if "caudal" in c.lower()][0]
caudal_month = (caudal_df
    .groupby(["localizacao","ym"], as_index=False)
    .agg(caudal_medio_mensal=(_caudal_col,"mean"))
)

# - Meteo: tx, tn, ETo (média), rr (soma)
meteo_month = (meteo_df
    .groupby(["lat","long","ym"], as_index=False)
    .agg(
        tx_mean=("tx","mean"),
        tn_mean=("tn","mean"),
        eto_mean=("ETo","mean"),
        rr_sum=("rr","sum")
    )
)
# Construir chave de grid id para compatibilizar com nearest (2 casas)
meteo_month["eobs_grid"] = meteo_month["lat"].round(2).astype(str) + "," + meteo_month["long"].round(2).astype(str)

# Filtrar por estações/grids mais próximos
# Condutividade: usar a estação mais próxima de cada poço
# (juntar pela estação 'cond_station' e ficar com a coluna 'codigo' do poço)
condut_sel = (condut_month
    .rename(columns={"codigo":"cond_codigo"})
    .merge(piezo_keys[["codigo","cond_station"]], left_on="cond_codigo", right_on="cond_station", how="inner")
    .drop(columns=["cond_station","cond_codigo"]))

# Caudal: usar estação mapeada
caudal_sel = caudal_month.merge(piezo_keys[["codigo","caudal_station"]], left_on="localizacao", right_on="caudal_station", how="inner")
caudal_sel = caudal_sel.drop(columns=["caudal_station"]).rename(columns={"localizacao":"caudal_station"})

# Meteo: juntar por grid id
meteo_sel = meteo_month.merge(piezo_keys[["codigo","eobs_grid"]], on="eobs_grid", how="inner")
meteo_sel = meteo_sel.drop(columns=["lat","long"])  # não necessários depois do join

# Unificar por poço e mês
unified = piezo_month.merge(piezo_keys, on="codigo", how="left")
unified = unified.merge(condut_sel, on=["codigo","ym"], how="left")
unified = unified.merge(caudal_sel, on=["codigo","ym"], how="left")
unified = unified.merge(meteo_sel, on=["codigo","ym"], how="left")

# Limpeza de colunas duplicadas e ordenação antes de exportar
if "caudal_station_x" in unified.columns or "caudal_station_y" in unified.columns:
    unified["caudal_station"] = unified.get("caudal_station_y").combine_first(unified.get("caudal_station_x"))
    unified = unified.drop(columns=[c for c in ["caudal_station_x","caudal_station_y"] if c in unified.columns])
if "eobs_grid_x" in unified.columns or "eobs_grid_y" in unified.columns:
    unified["eobs_grid"] = unified.get("eobs_grid_y").combine_first(unified.get("eobs_grid_x"))
    unified = unified.drop(columns=[c for c in ["eobs_grid_x","eobs_grid_y"] if c in unified.columns])

# Reordenar colunas principais
order = [
    "codigo","ym","caudal_station","cond_station","eobs_grid",
    "profundidade_mediana","nivel_piezometrico_mediana","condutividade_median",
    "caudal_medio_mensal","tx_mean","tn_mean","eto_mean","rr_sum"
]
keep = [c for c in order if c in unified.columns]
unified = unified[keep + [c for c in unified.columns if c not in keep]]

# Exportar dataset mensal unificado
unified_csv = f"{base}/mensal_por_poco.csv"
unified.to_csv(unified_csv, index=False)
print(f"[Mensal] Linhas: {len(unified)} -> {unified_csv}")

# Função de heatmap de correlação por poço
try:
    import plotly.express as px
except Exception:
    px = None

def plot_heatmap_poco(poco_id: str, corr_method: str = "pearson"):
    cols = [
        "rr_sum",                  # precipitação (soma mensal)
        "profundidade_mediana",    # nível (profundidade)
        "eto_mean",                # evapotranspiração (média mensal)
        "caudal_medio_mensal",     # caudal (média mensal)
        "condutividade_median"     # condutividade (mediana mensal)
    ]
    df = unified[unified["codigo"].astype(str) == str(poco_id)][["ym"] + [c for c in cols if c in unified.columns]].copy()
    df = df.set_index("ym").sort_index()
    # remover colunas totalmente vazias
    df = df.dropna(axis=1, how="all")
    if df.shape[1] < 2:
        print("Poucas variáveis para correlacionar neste poço.")
        return None
    corr = df.corr(method=corr_method, min_periods=10)
    if px is None:
        import seaborn as sns, matplotlib.pyplot as plt
        ax = sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_title(f"Correlação ({corr_method}) - Poço {poco_id}")
        return ax
    fig = px.imshow(
        corr,
        text_auto=True,
        zmin=-1, zmax=1,
        color_continuous_scale="RdBu_r",
        title=f"Correlação ({corr_method}) - Poço {poco_id}"
    )
    fig.update_layout(height=600, width=800)
    return fig

# Guardar todos os heatmaps (Pearson) por poço
from pathlib import Path
out_dir_pearson = Path(base) / "heatmaps" / "pearson"
out_dir_pearson.mkdir(parents=True, exist_ok=True)
for pid in sorted(unified["codigo"].dropna().astype(str).unique().tolist()):
    fig = plot_heatmap_poco(pid, corr_method="pearson")
    if hasattr(fig, "write_image"):
        try:
            fig.write_image(str(out_dir_pearson / f"heatmap_pearson_{pid.replace('/', '_')}.png"), scale=2)
        except Exception:
            pass

# Widget opcional para interação
try:
    import ipywidgets as widgets
    from IPython.display import display
    poco_opts = sorted(unified["codigo"].dropna().astype(str).unique().tolist())
    dd = widgets.Dropdown(options=poco_opts, description="poço_id:")
    btn = widgets.Button(description="Mostrar heatmap")
    out = widgets.Output()
    def _on_click(_):
        out.clear_output()
        with out:
            fig = plot_heatmap_poco(dd.value)
            if hasattr(fig, 'show'):
                fig.show()
    btn.on_click(_on_click)
    display(widgets.HBox([dd, btn]), out)
except Exception:
    pass

/var/folders/f9/slpppqbj1fs9tjk3hnc6r70c0000gn/T/ipykernel_49034/415403569.py:24: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  piezo_df["ym"]  = piezo_df["data"].dt.to_period("M").dt.to_timestamp()
/var/folders/f9/slpppqbj1fs9tjk3hnc6r70c0000gn/T/ipykernel_49034/415403569.py:25: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  condut_df["ym"] = condut_df["data"].dt.to_period("M").dt.to_timestamp()
/var/folders/f9/slpppqbj1fs9tjk3hnc6r70c0000gn/T/ipykernel_49034/415403569.py:26: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  caudal_df["ym"] = caudal_df["data"].dt.to_period("M").dt.to_timestamp()


[Mensal] Linhas: 9327 -> /Users/diogopinto/Documents/Pessoal/path_4med/eobs_netcdf/usar_model/correlacao_matriz/mensal_por_poco.csv


Output()

####  7) Código – Heatmap de correlação (Spearman) por poço
Nesta célula:
- Reutilizamos o dataset mensal `unified` (da célula 6) para cada poço.
- Selecionamos as variáveis: `rr_sum`, `profundidade_mediana`, `eto_mean`, `caudal_medio_mensal`, `condutividade_median`.
- Calculamos a matriz de correlação de Spearman (associação monotónica) com `min_periods=10` meses válidos.
- Geramos um heatmap por poço (Plotly/Seaborn) e guardamos as imagens em `heatmaps/spearman/`.
- Inclui um widget opcional para escolher o poço e visualizar o heatmap.
Nota: Spearman é menos sensível a outliers e capta relações monotónicas não necessariamente lineares.


In [13]:
# 7) Heatmap de correlação (Spearman) por poço
# Reutiliza o dataset mensal 'unified' criado na célula 6
try:
    import plotly.express as px  # pode já estar carregado
except Exception:
    px = None

def plot_heatmap_poco_spearman(poco_id: str):
    cols = [
        "rr_sum",                  # precipitação
        "profundidade_mediana",    # nível (profundidade)
        "eto_mean",                # ETo
        "caudal_medio_mensal",     # caudal
        "condutividade_median"     # condutividade
    ]
    df = unified[unified["codigo"].astype(str) == str(poco_id)][["ym"] + [c for c in cols if c in unified.columns]].copy()
    df = df.set_index("ym").sort_index()
    df = df.dropna(axis=1, how="all")
    if df.shape[1] < 2:
        print("Poucas variáveis para correlacionar neste poço.")
        return None
    corr = df.corr(method="spearman", min_periods=10)
    if px is None:
        import seaborn as sns, matplotlib.pyplot as plt
        ax = sns.heatmap(corr, annot=True, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_title(f"Correlação (Spearman) - Poço {poco_id}")
        return ax
    fig = px.imshow(
        corr,
        text_auto=True,
        zmin=-1, zmax=1,
        color_continuous_scale="RdBu_r",
        title=f"Correlação (Spearman) - Poço {poco_id}"
    )
    fig.update_layout(height=600, width=800)
    return fig

# Guardar todos os heatmaps (Spearman) por poço
from pathlib import Path
out_dir_spearman = Path(base) / "heatmaps" / "spearman"
out_dir_spearman.mkdir(parents=True, exist_ok=True)
for pid in sorted(unified["codigo"].dropna().astype(str).unique().tolist()):
    fig = plot_heatmap_poco_spearman(pid)
    if hasattr(fig, "write_image"):
        try:
            fig.write_image(str(out_dir_spearman / f"heatmap_spearman_{pid.replace('/', '_')}.png"), scale=2)
        except Exception:
            pass

# Widget opcional para interação (Spearman)
try:
    import ipywidgets as widgets
    from IPython.display import display
    poco_opts = sorted(unified["codigo"].dropna().astype(str).unique().tolist())
    dd_s = widgets.Dropdown(options=poco_opts, description="poço_id:")
    btn_s = widgets.Button(description="Heatmap Spearman")
    out_s = widgets.Output()
    def _on_click_s(_):
        out_s.clear_output()
        with out_s:
            fig = plot_heatmap_poco_spearman(dd_s.value)
            if hasattr(fig, 'show'):
                fig.show()
    btn_s.on_click(_on_click_s)
    display(widgets.HBox([dd_s, btn_s]), out_s)
except Exception:
    pass


Output()

### 8) Plano das correlações

Nesta secção organizamos as correlações em duas partes:

- 8.1) Correlação caudal vs nível piezométrico mensal (`nivel_piezometrico_mediana`), com e sem lag de 1 mês; exportamos o CSV para `output/correlacoes/nivel/`.
- 8.2) Correlação caudal vs profundidade do nível de água mensal (`profundidade_mediana`), com e sem lag de 1 mês; exportamos o CSV para `output/correlacoes/profundidade/`.

Em ambas calculamos Pearson e Spearman, com `min_periods=10`, e juntamos a distância `dist_km` ao ponto de caudal associado a cada poço.


####  8.1) Código – Correlações caudal vs nível piezométrico (sem lag e lag1)
Nesta célula:
- Calculamos, por poço, as correlações entre `caudal_medio_mensal` e `nivel_piezometrico_mediana`, com `min_periods=10`.
- Repetimos a análise com `caudal_medio_mensal_lag1`.
- Juntamos `dist_km` para avaliar relação com distância ao ponto de caudal.
- Exportamos CSV para `output/correlacoes/nivel/correlacao_caudal_nivel_vs_distancia.csv`.


In [15]:
# 8.1) Correlação (caudal vs nível piezométrico) vs distância ao ponto de caudal
import numpy as np
import pandas as pd

# Precisamos das distâncias por poço ao ponto de caudal
from pathlib import Path
nearest_csv = f"{base}/distancias_outros_para_pocos.csv"
alt_nearest_csv = f"{base}/output/distancias_outros_para_pocos.csv"
csv_path = nearest_csv if Path(nearest_csv).exists() else alt_nearest_csv
nearest_df = pd.read_csv(csv_path)[["poco_id","dist_km_caudal"]].drop_duplicates("poco_id")
nearest_df = nearest_df.rename(columns={"poco_id":"codigo"})

# A partir do dataset mensal unificado já existente (célula 6): 'unified'
# Calcula correlação por poço entre 'caudal_medio_mensal' e 'nivel_piezometrico_mediana'

def corr_per_well(df: pd.DataFrame, method: str = "pearson", min_periods: int = 10):
    out_rows = []
    for codigo, g in df.groupby("codigo"):
        sub = g[["caudal_medio_mensal","nivel_piezometrico_mediana"]].dropna()
        if sub.shape[0] >= min_periods:
            r = sub["caudal_medio_mensal"].corr(sub["nivel_piezometrico_mediana"], method=method)
        else:
            r = np.nan
        out_rows.append({
            "codigo": codigo,
            f"corr_{method}": r,
            "n_obs": sub.shape[0]
        })
    return pd.DataFrame(out_rows)

# Criar série de caudal com lag de 1 mês por poço
unified_sorted = unified.sort_values(["codigo","ym"]).copy()
unified_sorted["caudal_medio_mensal_lag1"] = (
    unified_sorted.groupby("codigo")["caudal_medio_mensal"].shift(1)
)

# Correlações sem lag
corr_pearson = corr_per_well(unified_sorted, method="pearson")
corr_spearman = corr_per_well(unified_sorted, method="spearman")
nonlag = corr_pearson.merge(corr_spearman, on=["codigo","n_obs"], how="outer")

# Correlações com lag de 1 mês (usar coluna lag como se fosse caudal)
_tmp = unified_sorted[["codigo","ym","caudal_medio_mensal_lag1","nivel_piezometrico_mediana"]].rename(
    columns={"caudal_medio_mensal_lag1":"caudal_medio_mensal"}
)
corr_pearson_lag1 = corr_per_well(_tmp, method="pearson").rename(
    columns={"corr_pearson":"corr_pearson_lag1", "n_obs":"n_obs_lag1"}
)
corr_spearman_lag1 = corr_per_well(_tmp, method="spearman").rename(
    columns={"corr_spearman":"corr_spearman_lag1", "n_obs":"n_obs_lag1"}
)

# Juntar tudo e distâncias
summary = (nonlag
    .merge(corr_pearson_lag1[["codigo","corr_pearson_lag1","n_obs_lag1"]], on="codigo", how="left")
    .merge(corr_spearman_lag1[["codigo","corr_spearman_lag1","n_obs_lag1"]], on=["codigo","n_obs_lag1"], how="left")
    .merge(nearest_df, on="codigo", how="left")
    .rename(columns={"dist_km_caudal":"dist_km"})
    .sort_values("dist_km")
)
# manter compatibilidade: 'n_obs' refere-se ao sem lag

# Exportar tabela com colunas de lag (nível)
from pathlib import Path
nivel_dir = Path(base) / "output" / "correlacoes" / "nivel"
nivel_dir.mkdir(parents=True, exist_ok=True)
out_csv = str(nivel_dir / "correlacao_caudal_nivel_vs_distancia.csv")
summary.to_csv(out_csv, index=False)
print(f"[Export] {len(summary)} linhas -> {out_csv}")

# Scatter interativo (mantém Pearson sem lag); opcionalmente plotar lag1 noutro gráfico
try:
    import plotly.express as px
    fig = px.scatter(summary, x="dist_km", y="corr_pearson", hover_data=["codigo","n_obs","corr_pearson_lag1","n_obs_lag1"],
                     trendline="ols", title="Correlação (Pearson) caudal vs nível vs distância (sem lag)")
    fig.update_layout(xaxis_title="Distância ao ponto de caudal (km)", yaxis_title="Correlação Pearson (sem lag)")
    fig.show()
except Exception:
    try:
        import seaborn as sns, matplotlib.pyplot as plt
        ax = sns.regplot(data=summary, x="dist_km", y="corr_pearson")
        ax.set_title("Correlação (Pearson) caudal vs nível vs distância (sem lag)")
        ax.set_xlabel("Distância ao ponto de caudal (km)")
        ax.set_ylabel("Correlação Pearson (sem lag)")
        plt.show()
    except Exception:
        pass

print(f"Poços na tabela: {summary['codigo'].nunique()}")
summary


[Export] 73 linhas -> /Users/diogopinto/Documents/Pessoal/path_4med/eobs_netcdf/usar_model/correlacao_matriz/output/correlacoes/nivel/correlacao_caudal_nivel_vs_distancia.csv


Poços na tabela: 73


,codigo,corr_pearson,n_obs,corr_spearman,corr_pearson_lag1,n_obs_lag1,corr_spearman_lag1,dist_km
2,331/2,0.546339,84,0.593342,0.630599,83,0.598838,9.665531
1,330/183,0.236390,227,0.389634,0.193984,226,0.308184,10.183768
5,342/97,0.781947,62,0.769638,0.531421,61,0.409007,13.717825
0,329/6,0.517114,18,0.570986,0.496932,17,0.480687,13.797998
4,342/78,0.361422,96,0.401446,0.394290,95,0.425583,15.622714
...,...,...,...,...,...,...,...,...
71,444/318,0.197151,55,0.226604,0.243509,54,0.191816,98.736673
61,432/69,0.288296,21,0.124675,0.357052,20,0.249624,98.778941
70,444/317,0.179259,188,0.218021,0.229125,187,0.261206,98.800216
72,444/85,0.122209,143,0.248644,0.294513,142,0.381427,102.335929


####  8.2) Código – Correlações caudal vs profundidade (sem lag e lag1)
Nesta célula:
- Calculamos, por poço, as correlações entre `caudal_medio_mensal` e `profundidade_mediana`, exigindo `min_periods=10`.
- Repetimos com `caudal_medio_mensal_lag1` para avaliar resposta retardada (~1 mês).
- Agregamos número de observações e juntamos a distância `dist_km` ao ponto de caudal.
- Exportamos CSV para `output/correlacoes/profundidade/correlacao_caudal_profundidade_vs_distancia.csv`.
Teoria: comparando sem lag vs lag1 inferimos se a resposta do aquífero é mais imediata ou atrasada relativamente ao regime de caudais.


In [16]:
# 8.2) Correlação (caudal vs profundidade do nível de água)
import numpy as np
import pandas as pd
from pathlib import Path

# Preparar distâncias
nearest_csv = f"{base}/distancias_outros_para_pocos.csv"
alt_nearest_csv = f"{base}/output/distancias_outros_para_pocos.csv"
csv_path = nearest_csv if Path(nearest_csv).exists() else alt_nearest_csv
nearest_df2 = pd.read_csv(csv_path)[["poco_id","dist_km_caudal"]].drop_duplicates("poco_id").rename(columns={"poco_id":"codigo"})

# Função genérica para correlação por poço entre caudal e uma variável alvo
def corr_per_well_target(df: pd.DataFrame, target_col: str, method: str = "pearson", min_periods: int = 10):
    out_rows = []
    for codigo, g in df.groupby("codigo"):
        sub = g[["caudal_medio_mensal", target_col]].dropna()
        if sub.shape[0] >= min_periods:
            r = sub["caudal_medio_mensal"].corr(sub[target_col], method=method)
        else:
            r = np.nan
        out_rows.append({"codigo": codigo, f"corr_{method}": r, "n_obs": sub.shape[0]})
    return pd.DataFrame(out_rows)

# Preparar dataset (ordenado) e criar lag de caudal
unified_sorted2 = unified.sort_values(["codigo","ym"]).copy()
unified_sorted2["caudal_medio_mensal_lag1"] = unified_sorted2.groupby("codigo")["caudal_medio_mensal"].shift(1)

# 8.2.A) Sem lag (profundidade_mediana)
corr_p_sem = corr_per_well_target(unified_sorted2, target_col="profundidade_mediana", method="pearson")
corr_s_sem = corr_per_well_target(unified_sorted2, target_col="profundidade_mediana", method="spearman")
nonlag_depth = corr_p_sem.merge(corr_s_sem, on=["codigo","n_obs"], how="outer")

# 8.2.B) Lag 1 mês (usar caudal_medio_mensal_lag1)
_tmp2 = unified_sorted2[["codigo","ym","caudal_medio_mensal_lag1","profundidade_mediana"]].rename(columns={"caudal_medio_mensal_lag1":"caudal_medio_mensal"})
corr_p_lag1 = corr_per_well_target(_tmp2, target_col="profundidade_mediana", method="pearson").rename(columns={"corr_pearson":"corr_pearson_lag1", "n_obs":"n_obs_lag1"})
corr_s_lag1 = corr_per_well_target(_tmp2, target_col="profundidade_mediana", method="spearman").rename(columns={"corr_spearman":"corr_spearman_lag1", "n_obs":"n_obs_lag1"})

# Juntar e adicionar distâncias
summary_depth = (nonlag_depth
    .merge(corr_p_lag1[["codigo","corr_pearson_lag1","n_obs_lag1"]], on="codigo", how="left")
    .merge(corr_s_lag1[["codigo","corr_spearman_lag1","n_obs_lag1"]], on=["codigo","n_obs_lag1"], how="left")
    .merge(nearest_df2, on="codigo", how="left")
    .rename(columns={"dist_km_caudal":"dist_km"})
    .sort_values("dist_km")
)

# Exportar CSVs
prof_dir = Path(base) / "output" / "correlacoes" / "profundidade"
prof_dir.mkdir(parents=True, exist_ok=True)
out_csv_prof = str(prof_dir / "correlacao_caudal_profundidade_vs_distancia.csv")
summary_depth.to_csv(out_csv_prof, index=False)
print(f"[Export] {len(summary_depth)} linhas -> {out_csv_prof}")

# Mostrar tabela
summary_depth


[Export] 73 linhas -> /Users/diogopinto/Documents/Pessoal/path_4med/eobs_netcdf/usar_model/correlacao_matriz/output/correlacoes/profundidade/correlacao_caudal_profundidade_vs_distancia.csv


,codigo,corr_pearson,n_obs,corr_spearman,corr_pearson_lag1,n_obs_lag1,corr_spearman_lag1,dist_km
2,331/2,-0.546339,84,-0.593342,-0.630599,83,-0.598838,9.665531
1,330/183,-0.236390,227,-0.389634,-0.193984,226,-0.308184,10.183768
5,342/97,-0.781947,62,-0.769638,-0.531421,61,-0.409007,13.717825
0,329/6,-0.517114,18,-0.570986,-0.496932,17,-0.480687,13.797998
4,342/78,-0.361422,96,-0.401446,-0.394290,95,-0.425583,15.622714
...,...,...,...,...,...,...,...,...
71,444/318,-0.197151,55,-0.226604,-0.243509,54,-0.191816,98.736673
61,432/69,-0.288296,21,-0.124675,-0.357052,20,-0.249624,98.778941
70,444/317,-0.179259,188,-0.218021,-0.229125,187,-0.261206,98.800216
72,444/85,-0.122209,143,-0.248656,-0.294513,142,-0.381447,102.335929




#### Resultados da 8.2) – Explicação do CSV `correlacao_caudal_profundidade_vs_distancia.csv`
Este ficheiro resume, por poço, as correlações entre o caudal do rio e a profundidade do nível de água (`profundidade_mediana`), com e sem atraso de 1 mês, juntando a distância ao ponto de caudal usado.

- O que contém (por linha/poço)
  - `codigo`: identificador do poço.
  - `corr_pearson`: correlação de Pearson entre `caudal_medio_mensal` e `profundidade_mediana` (mesmo mês).
  - `n_obs`: número de meses usados na correlação sem lag (mínimo exigido: 10; abaixo disso fica vazio/NaN).
  - `corr_spearman`: correlação de Spearman para o mesmo par (mesmo mês).
  - `corr_pearson_lag1`: Pearson entre `caudal_medio_mensal` com lag de 1 mês e `profundidade_mediana`.
  - `n_obs_lag1`: número de meses usados na correlação com lag (mínimo 10).
  - `corr_spearman_lag1`: Spearman com lag de 1 mês.
  - `dist_km`: distância (km) entre o poço e o ponto de caudal utilizado.

- Como interpretar (recordando que a variável alvo é profundidade)
  - Sinal: negativo → quando o caudal aumenta, a profundidade diminui (o nível sobe, fica mais “superficial”); positivo → quando o caudal aumenta, a profundidade aumenta (nível desce).
  - Magnitude (regra prática): ~0.1 fraco; ~0.3 moderado‑fraco; ~0.5 moderado; ≥0.7 forte.
  - Pearson vs Spearman: diferenças grandes sugerem não‑linearidade/outliers; valores próximos → relação aproximadamente linear/estável.
  - Sem lag vs lag1: `corr_*_lag1` maior do que `corr_*` sugere resposta com atraso ~1 mês; se menor, resposta mais imediata.
  - `n_obs`/`n_obs_lag1`: atenção a amostras pequenas; NaN indica que não atingiu os 10 meses válidos.

- Verificações rápidas no ficheiro atual
  - `342/97`: `corr_pearson ≈ -0.78` (forte negativa) e `corr_pearson_lag1 ≈ -0.53` → relação sobretudo imediata (mesmo mês).
  - `331/2`: `corr_pearson ≈ -0.55` e `corr_pearson_lag1 ≈ -0.63` → melhora com lag1 → resposta com atraso.
  - `353/353`: NaN com `n_obs=9` → insuficiente (mínimo 10).
  - `365/71`: valores próximos de 0 → sem relação clara.

Este CSV é guardado em `output/correlacoes/profundidade/correlacao_caudal_profundidade_vs_distancia.csv`. Para análises adicionais, podes ordenar por `abs(corr_pearson)` ou pela diferença `corr_pearson_lag1 - corr_pearson` para destacar poços com resposta retardada.

### 9) Explorar diferenças entre correlações de Pearson e Spearman

Objetivo: identificar poços onde as correlações de Pearson e Spearman (caudal vs GWD) diferem mais e visualizar a nuvem de pontos para perceber a causa (outliers, não‑linearidade, etc.).

- Pode escolher a variável de GWD: `nivel_piezometrico_mediana` (8.1) ou `profundidade_mediana` (8.2).
- Pode escolher sem lag (mesmo mês) ou lag de 1 mês para o caudal.
- Mostramos uma tabela com os Top‑N poços pela diferença absoluta |Pearson − Spearman| e os respetivos gráficos de dispersão.


####  9) Código – Diferenças Pearson vs Spearman e scatter plots
Nesta célula:
- Calculamos, por poço, as correlações de Pearson e Spearman entre caudal e GWD  `profundidade_mediana`, com e sem lag de 1 mês.
- Listamos os Top‑N poços com maior diferença absoluta |Pearson−Spearman| para investigar potenciais outliers ou relações não‑lineares.
- Fornecemos um widget para escolher poço/variável/lag e visualizar a nuvem de pontos com linha de tendência.
Teoria: Pearson mede correlação linear; Spearman mede monotonicidade (ordens). Diferenças grandes sugerem não‑linearidade ou outliers influentes.


In [17]:
# 9) Poços com maior diferença |Pearson - Spearman| e nuvens de pontos
import numpy as np
import pandas as pd

try:
    import plotly.express as px
except Exception:
    px = None

# Dataset base para gráficos (mesmo objeto 'unified' da célula 6)
# unified: colunas ['codigo','ym','profundidade_mediana','nivel_piezometrico_mediana', 'caudal_medio_mensal', ...]

def build_diff_table(var: str = "nivel_piezometrico_mediana", use_lag: bool = False, top_n: int = 10):
    df = unified.sort_values(["codigo","ym"]).copy()
    if use_lag:
        df["caudal_for_corr"] = df.groupby("codigo")["caudal_medio_mensal"].shift(1)
    else:
        df["caudal_for_corr"] = df["caudal_medio_mensal"]
    rows = []
    for codigo, g in df.groupby("codigo"):
        sub = g[["caudal_for_corr", var]].dropna()
        if sub.shape[0] < 10:
            continue
        r_p = sub["caudal_for_corr"].corr(sub[var], method="pearson")
        r_s = sub["caudal_for_corr"].corr(sub[var], method="spearman")
        rows.append({
            "codigo": str(codigo),
            "pearson": r_p,
            "spearman": r_s,
            "abs_diff": np.abs(r_p - r_s),
            "n_obs": sub.shape[0],
        })
    out = pd.DataFrame(rows).sort_values("abs_diff", ascending=False).head(top_n)
    return out

# Mostrar top diffs para as duas variáveis e dois modos
top_n = 10
print("Top diffs (nível, sem lag):")
display(build_diff_table(var="nivel_piezometrico_mediana", use_lag=False, top_n=top_n))
print("Top diffs (nível, lag1):")
display(build_diff_table(var="nivel_piezometrico_mediana", use_lag=True, top_n=top_n))
print("Top diffs (profundidade, sem lag):")
display(build_diff_table(var="profundidade_mediana", use_lag=False, top_n=top_n))
print("Top diffs (profundidade, lag1):")
display(build_diff_table(var="profundidade_mediana", use_lag=True, top_n=top_n))

# Widget opcional para escolher poço/variável/lag e plotar scatter
try:
    import ipywidgets as widgets
    from IPython.display import display
    var_dd = widgets.Dropdown(options=["nivel_piezometrico_mediana","profundidade_mediana"], value="nivel_piezometrico_mediana", description="variável:")
    lag_dd = widgets.Dropdown(options=[("sem lag", False),("lag1", True)], value=False, description="caudal:")
    well_dd = widgets.Dropdown(options=sorted(unified["codigo"].astype(str).unique()), description="poço:")
    out = widgets.Output()

    def plot_scatter(codigo: str, var: str, use_lag: bool):
        g = unified[unified["codigo"].astype(str)==str(codigo)].sort_values("ym").copy()
        g["caudal_for_corr"] = g["caudal_medio_mensal"].shift(1) if use_lag else g["caudal_medio_mensal"]
        g = g[["ym","caudal_for_corr", var]].dropna()
        if g.shape[0] < 10:
            print("Poucos pontos depois de filtrar (n<10).")
            return
        title = f"Scatter {var} vs caudal ({'lag1' if use_lag else 'sem lag'}) - Poço {codigo}"
        if px is not None:
            fig = px.scatter(g, x="caudal_for_corr", y=var, hover_data=["ym"], trendline="ols", title=title)
            fig.update_layout(xaxis_title="caudal", yaxis_title=var)
            fig.show()
        else:
            import seaborn as sns, matplotlib.pyplot as plt
            ax = sns.regplot(data=g, x="caudal_for_corr", y=var, scatter_kws={"s":20, "alpha":0.7})
            ax.set_title(title)
            ax.set_xlabel("caudal")
            ax.set_ylabel(var)
            plt.show()

    def on_change(_):
        out.clear_output()
        with out:
            plot_scatter(well_dd.value, var_dd.value, lag_dd.value)

    btn = widgets.Button(description="Plotar")
    btn.on_click(on_change)
    display(widgets.HBox([well_dd, var_dd, lag_dd, btn]), out)
except Exception:
    pass


Top diffs (nível, sem lag):


,codigo,pearson,spearman,abs_diff,n_obs
31,404/62,0.369336,0.654703,0.285367,38
38,405/17,0.045667,0.280847,0.235180,267
34,405/12,-0.253885,-0.023452,0.230433,40
29,404/53,0.144052,0.373529,0.229477,105
46,405/7,0.257719,0.454342,0.196623,53
45,405/69,-0.048968,0.146632,0.195601,40
43,405/67,0.096639,0.271661,0.175022,253
44,405/68,0.019089,0.192209,0.173120,46
41,405/51,-0.179043,-0.014699,0.164344,117
57,432/69,0.288296,0.124675,0.163621,21


Top diffs (nível, lag1):


,codigo,pearson,spearman,abs_diff,n_obs
38,405/17,-0.205905,0.182402,0.388307,266
29,404/53,0.161842,0.542096,0.380253,104
31,404/62,0.547436,0.785964,0.238528,37
43,405/67,0.188898,0.361277,0.172379,252
41,405/51,-0.038941,0.129180,0.168121,117
19,390/114,0.354184,0.198653,0.155531,25
25,391/403,0.347086,0.197059,0.150028,16
46,405/7,0.412097,0.555881,0.143784,52
40,405/34,0.430255,0.571807,0.141552,128
44,405/68,0.026325,0.155240,0.128915,45


Top diffs (profundidade, sem lag):


,codigo,pearson,spearman,abs_diff,n_obs
31,404/62,-0.369336,-0.654703,0.285367,38
38,405/17,-0.045667,-0.280847,0.235180,267
34,405/12,0.253885,0.023452,0.230433,40
29,404/53,-0.144052,-0.373529,0.229477,105
46,405/7,-0.257719,-0.454342,0.196623,53
45,405/69,0.048968,-0.146632,0.195601,40
43,405/67,-0.096639,-0.271721,0.175082,253
44,405/68,-0.019089,-0.191475,0.172386,46
41,405/51,0.179043,0.014699,0.164344,117
57,432/69,-0.288296,-0.124675,0.163621,21


Top diffs (profundidade, lag1):


,codigo,pearson,spearman,abs_diff,n_obs
38,405/17,0.205905,-0.182402,0.388307,266
29,404/53,-0.161842,-0.542096,0.380253,104
31,404/62,-0.547436,-0.785964,0.238528,37
43,405/67,-0.188898,-0.361262,0.172364,252
41,405/51,0.038941,-0.129180,0.168121,117
19,390/114,-0.354184,-0.198653,0.155531,25
25,391/403,-0.347086,-0.197059,0.150028,16
46,405/7,-0.412097,-0.555881,0.143784,52
40,405/34,-0.430255,-0.571807,0.141552,128
44,405/68,-0.026325,-0.155240,0.128915,45


Output()

### 10) Modelação por poço (profundidade_mediana): seleção de submodelo e correção de autocorrelação

Nesta secção implementamos, para CADA poço, um processo em duas fases para explicar a profundidade do nível de água mensal (`profundidade_mediana`) a partir de variáveis hidrometeorológicas. Isto responde ao pedido do ponto 6 da professora: identificar, por poço, quais as variáveis que melhor explicam a variação do GWD e, depois, ajustar um modelo que tenha em conta a autocorrelação temporal dos resíduos.

O que a professora pediu (ponto 6):
- Ajustar um modelo linear entre GWD e “todas” as variáveis candidatas (caudal, meteo, condutividade, possivelmente com lags), e escolher o melhor submodelo (conjunto de preditores) por um critério objetivo.
- Em seguida, re‑ajustar o modelo final considerando que a série temporal tem autocorrelação (os resíduos não são independentes no tempo).

O que fazemos aqui, passo a passo:
1) Preparação dos dados por poço
   - Ordenamos a série mensal por `ym` e criamos `caudal_medio_mensal_lag1` (caudal do mês anterior) para permitir efeitos retardados.
   - Removemos linhas com valores em falta nas colunas necessárias. Exigimos pelo menos 10 meses úteis por poço.

2) Variável alvo e preditores candidatos
   - Alvo: `profundidade_mediana` (a tua variável principal).
   - Candidatos (se existirem no dataset): `caudal_medio_mensal`, `caudal_medio_mensal_lag1`, `rr_sum`, `eto_mean`, `tx_mean`, `tn_mean`, `condutividade_median`.
   - Nota: podes ajustar esta lista conforme a disponibilidade/qualidade dos dados.

3) Seleção do “melhor” submodelo (fase de descoberta)
   - Fazemos uma busca por subconjuntos de preditores até 4 variáveis (para evitar modelos demasiado complexos), e para cada subconjunto ajustamos uma regressão OLS.
   - Usamos o critério BIC para comparar modelos: menor BIC = melhor compromisso entre ajuste e parcimónia.
   - Guardamos o subconjunto com menor BIC (se nada funcionar, fazemos fallback para o melhor univariado por AIC).

4) Ajuste final com autocorrelação (fase de confirmação)
   - Com os preditores selecionados, ajustamos um modelo GLSAR com estrutura AR(1) (autocorrelação de 1ª ordem) para capturar dependência temporal nos resíduos.
   - Se o GLSAR falhar, caímos para OLS. Guardamos coeficientes e métricas (AIC/BIC) do ajuste final.

5) Saída e interpretação
   - Para cada poço, exportamos: `codigo`, `n_obs`, `vars` (lista de preditores escolhidos), método usado (`GLSAR_AR1` ou `OLS`), `aic`, `bic`, e coeficientes (`coef_const`, `coef_var1`, ...).
   - Onde encontrar: `output/modelos/profundidade/modelos_profundiade_bestsubset_glsar.csv`.
   - Leitura: `vars` indica as variáveis que melhor explicam a profundidade do nível; os `coef_...` dão o sinal e magnitude do efeito (controle de unidades!); BIC/AIC menores indicam melhor qualidade relativa.

Limitações e próximos passos
- A seleção é puramente estatística; validação temporal (ex. TimeSeriesSplit) pode reforçar robustez.
- Verificar colinearidade (VIF) para subconjuntos escolhidos.
- Poderás comparar este GLSAR AR(1) com SARIMAX (ARIMA com exógenas) para séries mais autocorrelacionadas/sazonais.
- Considerar incluir mais lags (p.ex. lag2) se justificável e houver dados suficientes.


####  10) Código – Ajuste de modelos por poço (profundidade_mediana)
Nesta célula implementamos, para cada poço:
- Geração do preditor `caudal_medio_mensal_lag1` (efeito retardado).
- Seleção do melhor submodelo (até 4 variáveis) via busca exaustiva minimizando o BIC entre candidatos: `caudal_medio_mensal`, `caudal_medio_mensal_lag1`, `rr_sum`, `eto_mean`, `tx_mean`, `tn_mean`, `condutividade_median` (apenas os que existirem no dataset do poço).
- Ajuste final com autocorrelação AR(1) usando GLSAR; caso falhe, usa OLS.
- Exporta um resumo por poço com variáveis escolhidas, método, AIC/BIC e coeficientes para `output/modelos/profundidade/modelos_profundiade_bestsubset_glsar.csv`.
Teoria: BIC penaliza a complexidade (número de preditores). GLSAR AR(1) modela resíduos com dependência temporal de 1ª ordem, comum em séries mensais.


In [20]:
# 10) Implementação: seleção de submodelo e ajuste final por poço (profundidade_mediana)
import itertools
import numpy as np
import pandas as pd
from pathlib import Path

import statsmodels.api as sm

# Preparar base com lag de caudal
unif = unified.sort_values(["codigo","ym"]).copy()
unif["caudal_medio_mensal_lag1"] = unif.groupby("codigo")["caudal_medio_mensal"].shift(1)

# Definir preditores candidatos (existentes)
all_candidates = [
    "caudal_medio_mensal",
    "caudal_medio_mensal_lag1",
    "rr_sum",
    "eto_mean",
    "tx_mean",
    "tn_mean",
    "condutividade_median",
]
existing_candidates = [c for c in all_candidates if c in unif.columns]

out_dir = Path(base) / "output" / "modelos" / "profundidade"
out_dir.mkdir(parents=True, exist_ok=True)

results_rows = []

for codigo, g in unif.groupby("codigo"):
    df = g[["ym","profundidade_mediana"] + existing_candidates].dropna().copy()
    if df.shape[0] < 10:
        continue
    df = df.sort_values("ym").reset_index(drop=True)
    y = df["profundidade_mediana"].astype(float)

    # Best-subset (até 4 preditores) por BIC
    best_bic = np.inf
    best_subset = []
    best_model = None

    max_k = min(4, len(existing_candidates))
    for k in range(1, max_k+1):
        for subset in itertools.combinations(existing_candidates, k):
            X = sm.add_constant(df[list(subset)].astype(float), has_constant='add')
            try:
                model = sm.OLS(y, X, missing='drop').fit()
                bic = model.bic
                if np.isfinite(bic) and bic < best_bic:
                    best_bic = bic
                    best_subset = list(subset)
                    best_model = model
            except Exception:
                continue

    if best_model is None:
        # fallback para univariado com menor AIC se nada couber
        fallback = None
        for c in existing_candidates:
            try:
                m = sm.OLS(y, sm.add_constant(df[[c]].astype(float))).fit()
                if fallback is None or m.aic < fallback.aic:
                    fallback = m; best_subset = [c]
            except Exception:
                pass
        best_model = fallback

    if best_model is None:
        continue

    # Ajuste final com GLSAR AR(1) usando os mesmos preditores
    try:
        X_best = sm.add_constant(df[best_subset].astype(float), has_constant='add')
        glsar = sm.GLSAR(y, X_best, rho=1)
        res_iter = glsar.iterative_fit(3)
        final_params = res_iter.params
        final_aic = res_iter.aic if hasattr(res_iter, 'aic') else np.nan
        final_bic = res_iter.bic if hasattr(res_iter, 'bic') else np.nan
        method = "GLSAR_AR1"
    except Exception:
        # fallback OLS
        res_iter = best_model
        final_params = best_model.params
        final_aic = best_model.aic
        final_bic = best_model.bic
        method = "OLS"

    # Guardar resultados resumidos
    results_rows.append({
        "codigo": str(codigo),
        "n_obs": int(df.shape[0]),
        "vars": "+".join(best_subset),
        "method": method,
        "aic": float(final_aic) if final_aic is not None else np.nan,
        "bic": float(final_bic) if final_bic is not None else np.nan,
        **{f"coef_{k}": float(v) for k, v in final_params.items()}
    })

summary_models = pd.DataFrame(results_rows).sort_values(["bic","aic","codigo"]) if results_rows else pd.DataFrame()

# Exportar
out_csv_models = str(out_dir / "modelos_profundiade_bestsubset_glsar.csv")
summary_models.to_csv(out_csv_models, index=False)
print(f"[Modelos] {len(summary_models)} poços modelados -> {out_csv_models}")

# Mostrar tabela
summary_models.head(20)


[Modelos] 7 poços modelados -> /Users/diogopinto/Documents/Pessoal/path_4med/eobs_netcdf/usar_model/correlacao_matriz/output/modelos/profundidade/modelos_profundiade_bestsubset_glsar.csv


,codigo,n_obs,vars,method,aic,bic,coef_const,coef_caudal_medio_mensal,coef_eto_mean,coef_tn_mean,coef_rr_sum,coef_tx_mean,coef_caudal_medio_mensal_lag1,coef_condutividade_median
0,377/94,10,caudal_medio_mensal+eto_mean+tn_mean,GLSAR_AR1,13.216477,14.005375,3.699043,-0.002214,-0.258163,0.111423,NaN,NaN,NaN,NaN
4,406/97,12,caudal_medio_mensal_lag1,GLSAR_AR1,14.761582,15.557373,27.454771,NaN,NaN,NaN,NaN,NaN,-0.008617,NaN
1,391/33,10,rr_sum+eto_mean+tx_mean+tn_mean,GLSAR_AR1,17.134230,18.120353,-0.191091,NaN,-0.513095,-0.371331,0.01003,0.593751,NaN,NaN
5,432/68,17,tx_mean,GLSAR_AR1,22.095525,23.640703,8.282823,NaN,NaN,NaN,NaN,0.063465,NaN,NaN
2,405/167,11,caudal_medio_mensal_lag1+eto_mean+tx_mean,GLSAR_AR1,28.405949,29.616290,13.480319,NaN,-1.106352,NaN,NaN,0.429606,-0.013837,NaN
6,444/317,11,caudal_medio_mensal+eto_mean+tn_mean+condutivi...,GLSAR_AR1,35.696629,37.209554,45.714204,-0.009688,-1.253818,0.308594,NaN,NaN,NaN,-0.104261
3,406/27,19,caudal_medio_mensal_lag1,GLSAR_AR1,38.325188,40.105931,23.422475,NaN,NaN,NaN,NaN,NaN,-0.000560,NaN


#### Métodos usados na 10) – O que são OLS, GLSAR(AR1), AIC e BIC

- OLS (Ordinary Least Squares)
  - Regressão linear clássica que estima coeficientes minimizando a soma dos erros quadráticos.
  - Supõe erros independentes, média zero e variância constante (homoscedasticidade).
  - Fornece coeficientes, erros‑padrão, AIC/BIC e métricas de ajuste.

- GLSAR com AR(1)
  - GLS = Generalized Least Squares: permite modelar estrutura de correlação nos resíduos.
  - AR(1): assume que os resíduos seguem \( e_t = \rho\, e_{t-1} + u_t \) (persistência temporal de 1ª ordem).
  - Útil em séries mensais com autocorrelação; reduz viés nos erros‑padrão e melhora a inferência.
  - No código, ajustamos OLS para seleção e depois re‑ajustamos o submodelo escolhido com GLSAR(AR1) via `iterative_fit`.

- Por que usamos AR(1)?
  - É o modelo de dependência temporal mais simples e frequente em séries hidrológicas/meterológicas mensais.
  - Captura a maior parte da persistência sem inflacionar demasiadamente a complexidade.

- Seleção de submodelo por BIC (best‑subset)
  - Testamos todos os subconjuntos de preditores até 4 variáveis e comparamos por BIC.
  - BIC = penaliza mais modelos com muitos parâmetros (parcimónia). Menor BIC indica melhor compromisso ajuste/complexidade.
  - Fluxo: para cada poço → gerar subconjuntos → ajustar OLS → escolher o de menor BIC → re‑ajustar com GLSAR(AR1).

- AIC vs BIC (intuição)
  - AIC ≈ qualidade preditiva (penalização mais leve), BIC ≈ preferência por modelos mais simples (penalização mais forte).
  - Ambos são relativos: comparam modelos candidatos sobre o mesmo conjunto de dados; valores menores são melhores (não têm escala absoluta).

- Interpretação dos resultados exportados
  - `vars`: preditores escolhidos (ex.: `caudal_medio_mensal_lag1+rr_sum`).
  - `method`: `GLSAR_AR1` se o ajuste final considerou autocorrelação; `OLS` se houve fallback.
  - `aic`/`bic`: do ajuste final; quanto menores, melhor (comparação entre poços só é indicativa; use com cautela).
  - `coef_const` e `coef_<variável>`: impacto marginal estimado na `profundidade_mediana` (atenção às unidades e ao sinal).

- Notas práticas
  - Requer nº mínimo de observações (>=10 após remoção de NaNs). Menos do que isso, o poço é ignorado.
  - Colinearidade entre preditores pode afetar a estabilidade dos coeficientes; considerar VIF/diagnósticos se necessário.
  - Poderás estender com mais lags (lag2, lag3) se houver dados suficientes, ou usar SARIMAX para sazonalidade forte.
